# Phase 1 - SQL Analysis Layer

## Healthcare AI System

#### Load raw data + writing some queries

In [31]:
# Imports

import pandas as pd
import numpy as np
import sqlite3
import os

In [32]:
# 1) load raw data

patients = pd.read_csv('../data/patients.csv')
visits = pd.read_csv('../data/visits.csv')
billing = pd.read_csv('../data/billing.csv') 


print('Patients shape:', patients.shape)
print('Visits shape:', visits.shape)
print('Billing shape:', billing.shape)

Patients shape: (5000, 7)
Visits shape: (25000, 8)
Billing shape: (25000, 7)


In [33]:
# 2) create a SQLite database and load the data into it
os.makedirs('../db', exist_ok=True)

conn = sqlite3.connect('../db/healthcare.db')
cursor = conn.cursor()

print('Database connected:', conn)

Database connected: <sqlite3.Connection object at 0x12444e5c0>


In [34]:
# 3) Load the data into the database
patients.to_sql('patients', conn, if_exists='replace', index=False)
visits.to_sql('visits', conn, if_exists='replace', index=False)
billing.to_sql('billing', conn, if_exists='replace', index=False)

print('Data loaded into the database.')


Data loaded into the database.


In [35]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

       name
0  patients
1    visits
2   billing


In [36]:
# Quick preview of tables
print('=== Patients (3 first rows) Table ===')
print(pd.read_sql('SELECT * FROM patients LIMIT 3;', conn).to_string(index=False))

print('\n=== Visits (3 first rows) Table ===')
print(pd.read_sql('SELECT * FROM visits LIMIT 3;', conn).to_string(index=False))

print('\n=== Billing (3 first rows) Table ===')
print(pd.read_sql('SELECT * FROM billing LIMIT 3;', conn).to_string(index=False))

=== Patients (3 first rows) Table ===
 patient_id  age gender      city insurance_provider  chronic_flag registration_date
          1   53      M Hyderabad         SecureLife             0        2025-05-14
          2   42      M      Pune         HealthPlus             0        2025-11-18
          3   56      F Hyderabad         HealthPlus             0        2025-05-11

=== Visits (3 first rows) Table ===
 visit_id  patient_id visit_date  department visit_type  length_of_stay_hours risk_score  doctor_id
        1         756 2025-10-18  Cardiology         ER                  3.48        Low        169
        2        4102 2025-04-06 Orthopedics        OPD                 15.31       High        148
        3        2964 2025-07-13         ICU         ER                 34.36        Low        153

=== Billing (3 first rows) Table ===
 bill_id  visit_id  billed_amount  approved_amount claim_status  payment_days billing_date
       1         1       23577.37             0.00     R

In [40]:
dept_workload = pd.read_sql("""
SELECT
    department,
    COUNT(visit_id) AS num_visits,
    ROUND(AVG(length_of_stay_hours), 2) AS avg_length_of_stay,
    ROUND(MAX(length_of_stay_hours), 2) AS max_length_of_stay,
    SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) AS high_risk_visits
FROM visits
GROUP BY department
ORDER BY 2 DESC;
""", conn)

print(dept_workload.to_string(index=False))

 department  num_visits  avg_length_of_stay  max_length_of_stay  high_risk_visits
    General        4228               19.43               78.42               839
         ER        4220               19.53               74.81               872
  Neurology        4165               19.72               70.12               846
Orthopedics        4164               19.66               71.60               842
 Cardiology        4159               19.60               77.42               790
        ICU        4064               19.36               72.47               845


In [43]:
doctor_workload = pd.read_sql("""
SELECT
    doctor_id,
    COUNT(visit_id) AS num_visits,
    SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) AS high_risk_cases,
    ROUND(SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END)*100.0 / COUNT(visit_id), 1) AS high_risk_pct
FROM visits
GROUP BY doctor_id
ORDER BY 3 DESC
LIMIT 10;
""",conn)

print(doctor_workload.to_string(index=False))

 doctor_id  num_visits  high_risk_cases  high_risk_pct
       174         264               71           26.9
       198         252               69           27.4
       169         279               68           24.4
       177         266               67           25.2
       135         261               65           24.9
       105         250               65           26.0
       188         285               64           22.5
       180         290               64           22.1
       131         266               62           23.3
       178         245               61           24.9


In [44]:
patient_visists = pd.read_sql("""
SELECT 
    CASE 
        WHEN num_visits = 1 THEN '1 visit'
        WHEN num_visits BETWEEN 2 AND 3 THEN '2-4 visits'
        WHEN num_visits BETWEEN 4 AND 5 THEN '4-5 visits'
        ELSE '6+ visits'
    END AS visit_frequency_bucket,
    COUNT(patient_id) AS num_patients
FROM
(
    SELECT
        patient_id, 
        count(visit_id) AS num_visits
    FROM visits 
    GROUP BY patient_id
)
GROUP BY 1
ORDER BY 2 DESC;
""", conn)

print(patient_visists.to_string(index=False))

visit_frequency_bucket  num_patients
             6+ visits          1937
            4-5 visits          1731
            2-4 visits          1150
               1 visit           149


In [48]:
risk_score_distribution = pd.read_sql("""
SELECT 
    department,
    sum(CASE WHEN risk_score = 'Low' THEN 1 ELSE 0 END)    AS low,
    sum(CASE WHEN risk_score = 'Medium' THEN 1 ELSE 0 END) AS medium,
    sum(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END)   AS high,
    count(*) AS total
FROM visits
GROUP BY department
ORDER BY high DESC;
""", conn)

print(risk_score_distribution.to_string(index=False))

 department  low  medium  high  total
         ER 2092    1256   872   4220
  Neurology 2058    1261   846   4165
        ICU 2037    1182   845   4064
Orthopedics 2078    1244   842   4164
    General 2123    1266   839   4228
 Cardiology 2082    1287   790   4159


In [52]:
insurance_distribution = pd.read_sql("""
SELECT
    p.insurance_provider,
    count(b.bill_id) AS num_claims,
    round(sum(b.billed_amount), 0) AS total_billed,
    round(AVG(b.billed_amount), 0) AS avg_billed,
    round(sum(approved_amount), 0) AS total_approved,
    round(AVG(approved_amount), 0) AS avg_approved,
    sum(case when claim_status = 'Paid' then 1 else 0 end) AS paid,
    sum(case when claim_status = 'Pending' then 1 else 0 end) AS pending,
    sum(case when claim_status = 'Rejected' then 1 else 0 end) AS rejected
FROM billing b
JOIN visits v ON b.visit_id = v.visit_id
JOIN patients p ON v.patient_id = p.patient_id
group by p.insurance_provider
order by total_billed desc;
""", conn)

print(insurance_distribution.to_string(index=False))

insurance_provider  num_claims  total_billed  avg_billed  total_approved  avg_approved  paid  pending  rejected
         MediCareX        6532   134591163.0     20605.0     100135469.0       16156.0  3875     1661       996
           CareOne        6283   130707993.0     20803.0      96997758.0       16305.0  3787     1562       934
        HealthPlus        6220   130180741.0     20929.0      96251775.0       16389.0  3680     1609       931
        SecureLife        5965   126289040.0     21172.0      93770886.0       16561.0  3598     1431       936


In [54]:
rejected_insurance = pd.read_sql("""
SELECT
    p.insurance_provider,
    count(b.bill_id) AS num_claims,
    round(sum(case when claim_status = 'Rejected' then 1 else 0 end) * 100.0 / count(b.bill_id), 1) AS rejected_pct
FROM billing b
JOIN visits v ON b.visit_id = v.visit_id
JOIN patients p ON v.patient_id = p.patient_id
group by p.insurance_provider
order by 3 desc;
""", conn)

print(rejected_insurance.to_string(index=False))

insurance_provider  num_claims  rejected_pct
        SecureLife        5965          15.7
         MediCareX        6532          15.2
        HealthPlus        6220          15.0
           CareOne        6283          14.9


In [56]:
revenue_realization = pd.read_sql("""
SELECT
    p.insurance_provider,
    round(sum(b.billed_amount), 0) AS total_billed,
    round(sum(approved_amount), 0) AS total_approved,
    round(sum(approved_amount) * 100.0 / sum(b.billed_amount), 1) AS revenue_realization_pct
FROM billing b
JOIN visits v ON b.visit_id = v.visit_id
JOIN patients p ON v.patient_id = p.patient_id
WHERE approved_amount IS NOT NULL
group by p.insurance_provider
order by 4 desc;
""", conn)

print(revenue_realization.to_string(index=False))

insurance_provider  total_billed  total_approved  revenue_realization_pct
           CareOne   123701662.0      96997758.0                     78.4
        HealthPlus   122873457.0      96251775.0                     78.3
         MediCareX   128114983.0     100135469.0                     78.2
        SecureLife   120079039.0      93770886.0                     78.1


In [58]:
# checking missing values in all tables

for table in ['patients', 'visits', 'billing']:
    df = pd.read_sql(f'SELECT * FROM {table};', conn)
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f'\nMissing values in {table} table:')
        print(nulls.to_string())
        print()
    else:
        print(f'\nNo missing values in {table} table.')
    


No missing values in patients table.

No missing values in visits table.

Missing values in billing table:
approved_amount    1318
payment_days        790



In [59]:
# checking duplicates 
duplicate_patients = pd.read_sql("""
SELECT 
    patient_id, 
    COUNT(*) AS count
FROM patients
GROUP BY patient_id
HAVING COUNT(*) > 1;
""", conn)

if len(duplicate_patients) == 0:
    print('\nNo duplicate patient IDs found.')
else:
    print('\nDuplicate patient IDs found:')
    print(duplicate_patients.to_string(index=False))


No duplicate patient IDs found.


# Export model_table.csv

In [60]:
# Join all three tables into one flat table for ML modeling.
# This is the single source of truth for Phase 2 EDA and Phase 3 Modeling.
model_table = pd.read_sql("""
    SELECT
        -- Patient features
        p.patient_id,
        p.age,
        p.gender,
        p.city,
        p.insurance_provider,
        p.chronic_flag,
        p.registration_date,

        -- Visit features
        v.visit_id,
        v.visit_date,
        v.department,
        v.visit_type,
        v.length_of_stay_hours,
        v.risk_score,
        v.doctor_id,

        -- Billing features
        b.bill_id,
        b.billed_amount,
        b.approved_amount,
        b.claim_status,
        b.payment_days,
        b.billing_date

    FROM visits v
    JOIN patients p ON v.patient_id = p.patient_id
    JOIN billing  b ON v.visit_id   = b.visit_id
    ORDER BY v.visit_date
""", conn)

print("model_table shape:", model_table.shape)
print("Columns:", model_table.columns.tolist())
model_table.head(3)

model_table shape: (25000, 20)
Columns: ['patient_id', 'age', 'gender', 'city', 'insurance_provider', 'chronic_flag', 'registration_date', 'visit_id', 'visit_date', 'department', 'visit_type', 'length_of_stay_hours', 'risk_score', 'doctor_id', 'bill_id', 'billed_amount', 'approved_amount', 'claim_status', 'payment_days', 'billing_date']


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,bill_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,3993,9,M,Hyderabad,HealthPlus,1,2025-10-18,684,2025-01-20,General,ICU,5.79,Low,137,684,36721.68,36721.68,Paid,19.0,2025-08-15
1,76,59,M,Delhi,HealthPlus,0,2025-07-09,1412,2025-01-20,General,ER,34.80,Medium,109,1412,8365.47,4189.20,Pending,26.0,2025-11-14
2,3393,43,M,Hyderabad,HealthPlus,0,2025-07-10,1510,2025-01-20,ICU,ICU,31.37,Medium,135,1510,16529.35,0.00,Rejected,7.0,2025-04-24


In [62]:
# save the model table
import os
os.makedirs('../outputs', exist_ok=True)
model_table.to_csv('../outputs/model_table.csv', index=False)

print("model_table saved to ../outputs/model_table.csv")
print("model_table shape:", model_table.shape)
print("size of model_table.csv:", round(os.path.getsize('../outputs/model_table.csv')/1024,1), "KB")

model_table saved to ../outputs/model_table.csv
model_table shape: (25000, 20)
size of model_table.csv: 3102.2 KB


In [63]:
# close the database connection
conn.close()

print("Database connection closed.")

Database connection closed.
